In [12]:
input_folder = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses/"
file_name_list_csv_output_path = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/file_name_list.csv"
common_mzs_csv_output_path = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters_improved.csv"


In [13]:
import os
import pandas as pd
import scanpy as sc
import numpy as np


In [14]:
file_name_list = []
for file_name in os.listdir(input_folder):
    if file_name.endswith(".h5ad"):
        file_name_list.append(file_name)
file_name_list.sort()
file_name_list

['AFM_AD_1_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'AFM_AD_2_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'AFM_AD_3_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'AFM_AD_4_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'AFM_C_1_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'AFM_C_2_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'AFM_C_3_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'AFM_C_4_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'YFM_AD_1_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'YFM_AD_2_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'YFM_AD_3_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'YFM_AD_4_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'YFM_C_1_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'YFM_C_2_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'YFM_C_3_peaks_0.00001_top1000_9w_3p_5points.h5ad',
 'YFM_C_4_peaks_0.00001_top1000_9w_3p_5points.h5ad']

In [15]:
pd.DataFrame(file_name_list, columns=["file_name"]).to_csv(file_name_list_csv_output_path, index=False)

In [16]:
all_data = []

for file_name in file_name_list:
    # Split by underscores
    parts = file_name.split("_")

    # Extract pieces
    group_name = file_name[0]+ parts[1]         # Group name (first character + second part)
    sample_name = parts[2]                      # Third part

    # Construct full file path
    file_path = os.path.join(input_folder, file_name)

    # Read the AnnData object
    adata = sc.read_h5ad(file_path)
    
    # Extract mzs
    mzs_list = adata.var['mzs'].tolist()
    
    # Append to all_data
    for mz in mzs_list:
        all_data.append({
            "mzs": mz,
            "group": group_name,
            "sample": sample_name
        })
    
# Create DataFrame
df = pd.DataFrame(all_data)

In [17]:
df

,mzs,group,sample
0,324.2207,AAD,1
1,326.1996,AAD,1
2,337.1939,AAD,1
3,338.1985,AAD,1
4,339.2315,AAD,1
...,...,...,...
15995,1261.4266,YC,4
15996,1283.9533,YC,4
15997,1296.9355,YC,4
15998,1297.9411,YC,4


In [18]:
tolerance = 0.01

# Sort by group and mzs
df_sorted = df.sort_values(["group", "mzs"]).reset_index(drop=True)

# List to store matched clusters
matches = []

# Process per group
for group, g_df in df_sorted.groupby("group"):
    mz_values = g_df["mzs"].values
    
    for i, mz in enumerate(mz_values):
        # Find all m/z within ± tolerance
        mask = np.abs(mz_values - mz) <= tolerance
        cluster = g_df[mask]
        
        if len(cluster) > 3:  # Keep only if more than 1 match
            matches.append(cluster)

# Combine into one DataFrame
common_in_each_group_df = pd.concat(matches).drop_duplicates()

print(f"Found {len(common_in_each_group_df)} rows with same group and mz ±{tolerance}")
print(common_in_each_group_df.head())

Found 13872 rows with same group and mz ±0.01
        mzs group sample
0  324.2207   AAD      1
1  324.2207   AAD      2
2  324.2207   AAD      3
3  324.2207   AAD      4
4  326.1996   AAD      1


In [19]:
common_in_each_group_df

,mzs,group,sample
0,324.2207,AAD,1
1,324.2207,AAD,2
2,324.2207,AAD,3
3,324.2207,AAD,4
4,326.1996,AAD,1
...,...,...,...
15906,1020.6178,YC,2
15934,1088.8161,YC,1
15935,1088.8167,YC,3
15936,1088.8250,YC,4


In [20]:
tolerance = 0.01

# Sort by mzs
df_sorted = common_in_each_group_df.sort_values("mzs").reset_index(drop=True)
mz_values = df_sorted["mzs"].values

clusters = []
used_indices = set()

for i, mz in enumerate(mz_values):
    if i in used_indices:
        continue  # skip if already clustered
    
    # Find all mz within tolerance ±0.01
    mask = np.abs(mz_values - mz) <= tolerance
    cluster_indices = np.where(mask)[0]
    
    # Only consider clusters with more than 1 member
    if len(cluster_indices) > 15:
        # Add indices to used set to avoid duplicate clustering
        used_indices.update(cluster_indices)
        
        cluster = df_sorted.iloc[cluster_indices].copy()  # make a copy to avoid SettingWithCopyWarning
        # Assign common group name as the mz of the first member in this cluster (rounded nicely)
        cluster_name = f"{cluster.iloc[0]['mzs']:.4f}"
        cluster['common_group_name'] = cluster_name
        
        clusters.append(cluster)

# Combine all clusters into one DataFrame
common_mz_clusters = pd.concat(clusters).drop_duplicates().reset_index(drop=True)

print(f"Found {len(common_mz_clusters)} rows in clusters with mz ±{tolerance} containing >15 member")
print(common_mz_clusters.head())

Found 8448 rows in clusters with mz ±0.01 containing >15 member
        mzs group sample common_group_name
0  326.1934    YC      1          326.1934
1  326.1937    AC      3          326.1934
2  326.1937    AC      4          326.1934
3  326.1937    AC      2          326.1934
4  326.1937    YC      3          326.1934


In [21]:
common_mz = set(common_mz_clusters['common_group_name'].astype(float).tolist())
common_mz 

{326.1934,
 337.1867,
 339.2243,
 340.2283,
 341.2403,
 351.1655,
 351.2252,
 352.2294,
 353.2401,
 354.2482,
 355.2558,
 355.3677,
 355.8122,
 356.259,
 357.1526,
 357.2633,
 358.2651,
 360.133,
 363.1673,
 364.1692,
 365.1812,
 367.2559,
 368.2599,
 370.2409,
 370.2761,
 371.2513,
 372.255,
 373.1476,
 379.1624,
 380.1685,
 381.1773,
 382.1812,
 383.1934,
 383.2528,
 384.1968,
 385.2086,
 385.2683,
 386.2114,
 386.2384,
 387.246,
 388.2502,
 389.261,
 397.1724,
 399.2472,
 400.3386,
 403.2417,
 405.1758,
 411.2479,
 412.2465,
 412.2774,
 413.256,
 426.3549,
 431.241,
 455.2893,
 456.292,
 457.3038,
 458.3088,
 471.2829,
 478.3266,
 482.277,
 483.2836,
 484.2924,
 485.2983,
 486.3065,
 487.2775,
 491.3585,
 492.3619,
 496.339,
 497.3401,
 498.3334,
 499.2785,
 499.3409,
 500.2856,
 500.3106,
 500.3481,
 501.2947,
 501.3523,
 502.2982,
 502.3259,
 502.361,
 503.3643,
 504.3418,
 506.358,
 507.2679,
 514.3268,
 515.274,
 515.3307,
 516.3417,
 517.2879,
 517.3469,
 518.3205,
 522.353,
 5

In [22]:
common_mz_clusters["mzs"] = common_mz_clusters["mzs"].round(4)
common_mz_clusters.to_csv(common_mzs_csv_output_path, index=False)